# INST326 — Week 7 Exercises: Error Handling & Testing (Library Management)

**Focus (Week 7 only):** exception handling (`try` / `except` / `else` / `finally`), raising exceptions, defining simple custom exceptions, and **basic** unit testing with `unittest` (no fixtures beyond `setUp`/`tearDown`).

**Out of scope:** Anything introduced in Week 8 or later (e.g., inheritance, abstract classes, polymorphism, advanced testing/CI, design patterns).

> Context: Use a simple Library Management domain—books, members, catalog, and loans—to complete the tasks.


### Starter Helpers (Optional)

You may use or modify the minimal scaffolding below in your solutions. It intentionally avoids Week 8+ concepts.


In [ ]:
# Minimal, Week-7-safe scaffolding (no inheritance/ABCs).
from __future__ import annotations
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Optional, Dict, List
import json

class LibraryError(Exception):
    """Base library-related error for Week 7."""

class DuplicateBookError(LibraryError):
    pass

class OverdueLoanError(LibraryError):
    pass

@dataclass
class Book:
    isbn: str
    title: str
    copies: int = 1

@dataclass
class Member:
    member_id: str
    email: str

@dataclass
class Loan:
    isbn: str
    member_id: str
    due_date: datetime
    returned: bool = False

    def check_overdue(self) -> bool:
        if datetime.now() > self.due_date and not self.returned:
            return True
        return False

class Catalog:
    def __init__(self):
        self._books: Dict[str, Book] = {}

    def add_book(self, book: Book) -> None:
        if book.isbn in self._books:
            raise DuplicateBookError(f"ISBN already exists: {book.isbn}")
        if book.copies < 0:
            raise ValueError("copies must be non-negative")
        self._books[book.isbn] = book

    def get_book(self, isbn: str) -> Optional[Book]:
        return self._books.get(isbn)

    def load_from_json(self, path: str) -> int:
        # Intentionally minimal; implement robust handling in exercises.
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        count = 0
        for item in data:
            self.add_book(Book(isbn=item["isbn"], title=item["title"], copies=item.get("copies", 1)))
            count += 1
        return count


## 1) Validate ISBN with exceptions

Write a function `validate_isbn(isbn: str) -> str` that:
- Strips hyphens/spaces
- Verifies it is 10 or 13 digits (numeric only after stripping)
- Raises `ValueError` with a helpful message when invalid
Return the normalized ISBN string when valid.

In [ ]:
# Your code here
def validate_isbn(isbn: str) -> str:
    """
    Strips hyphens/spaces, verifies 10 or 13 digits, returns normalized ISBN.
    Raises ValueError if invalid.
    """
    # Strip hyphens and spaces
    normalized = isbn.replace("-", "").replace(" ", "")
    
    # Check if numeric and correct length
    if not normalized.isdigit():
        raise ValueError(f"ISBN must contain only digits: {isbn}")
    
    if len(normalized) not in (10, 13):
        raise ValueError(f"ISBN must be 10 or 13 digits, got {len(normalized)}: {isbn}")
    
    return normalized

# Example:
# print(validate_isbn("978-1-4028-9462-6"))  # -> '9781402894626'

## 2) Safe integer input for copies

Implement `parse_copies(text: str) -> int` that parses a positive integer for number of copies.
Use `try/except` to catch `ValueError`, and raise your own `ValueError` with a user-friendly message.

In [ ]:
def parse_copies(text: str) -> int:
    """
    Parses a positive integer for number of copies.
    Raises ValueError with user-friendly message if invalid.
    """
    try:
        copies = int(text)
        if copies <= 0:
            raise ValueError("copies must be a positive integer")
        return copies
    except ValueError:
        raise ValueError("copies must be a positive integer")

# parse_copies("3") -> 3
# parse_copies("three") -> ValueError('copies must be a positive integer')

## 3) Custom exception: OverdueLoanError

Extend `Loan.check_overdue()` to **raise** `OverdueLoanError` if the book is overdue (instead of returning `True/False`). 
Catch this exception in a separate function `assert_not_overdue(loan: Loan)` that returns `True` when ok and `False` when overdue.

In [ ]:
# Your code here
def assert_not_overdue(loan: Loan) -> bool:
    """
    Returns True if loan is not overdue, False if it is overdue.
    Catches OverdueLoanError from loan.check_overdue().
    """
    try:
        loan.check_overdue()
        return True  # No exception means not overdue
    except OverdueLoanError:
        return False  # Exception means overdue
# Tip: modify Loan.check_overdue or wrap/extend without Week 8+ concepts.

## 4) Robust date parsing

Write `parse_date(text: str) -> datetime` that accepts `YYYY-MM-DD`. If parsing fails, raise `ValueError('invalid date: ...')`.
Use `try/except` around `datetime.strptime`.

In [ ]:
# Your code here
def parse_date(text: str) -> datetime:
    """
    Accepts YYYY-MM-DD format. Raises ValueError if parsing fails.
    """
    try:
        return datetime.strptime(text, "%Y-%m-%d")
    except ValueError:
        raise ValueError(f"invalid date: {text}")
# Example:
# parse_date("2025-10-27")

## 5) Late fee calculation with ZeroDivisionError guard

Define `per_day_late_fee(total_fee: float, days_late: int) -> float` that computes `total_fee / days_late`.
If `days_late` is 0, raise `ZeroDivisionError` with a clear message. Show a `try/except` usage example that prints a friendly message instead of crashing.

In [ ]:
# Your code here
def per_day_late_fee(total_fee: float, days_late: int) -> float:
    """
    Computes total_fee / days_late.
    Raises ZeroDivisionError if days_late is 0.
    """
    if days_late == 0:
        raise ZeroDivisionError("Cannot calculate per-day fee: days_late is 0")
    return total_fee / days_late
# try/except demo here
def demo_late_fee():
    try:
        fee = per_day_late_fee(10.0, 0)
        print(f"Fee per day: ${fee:.2f}")
    except ZeroDivisionError as e:
        print(f"Error calculating fee: {e}")

## 6) Always-close file with try/finally

Implement `read_file_head(path: str, n: int=3) -> list[str]` that opens a UTF-8 text file and returns the first `n` lines (stripped).
Use explicit `try/finally` to ensure closing (even though `with` is preferred) to practice `finally` behavior.

In [ ]:
# Your code here
def read_file_head(path: str, n: int = 3) -> list[str]:
    """
    Opens a UTF-8 text file and returns first n lines (stripped).
    Uses try/finally to ensure file closure.
    """
    f = None
    try:
        f = open(path, "r", encoding="utf-8")
        lines = []
        for i, line in enumerate(f):
            if i >= n:
                break
            lines.append(line.strip())
        return lines
    finally:
        if f is not None:
            f.close()
    ...

## 7) Safe JSON catalog loading

Implement `safe_load_catalog(path: str, catalog: Catalog) -> int` that handles:
- `FileNotFoundError` → return 0
- `json.JSONDecodeError` → return 0
- `DuplicateBookError` → skip duplicate and continue
Return the number of **new** books added.

In [ ]:
# Your code here
import json

def safe_load_catalog(path: str, catalog: Catalog) -> int:
    """
    Handles FileNotFoundError, JSONDecodeError, and DuplicateBookError.
    Returns number of new books added.
    """
    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        return 0
    except json.JSONDecodeError:
        return 0
    
    count = 0
    for item in data:
        try:
            book = Book(isbn=item["isbn"], title=item["title"], 
                       copies=item.get("copies", 1))
            catalog.add_book(book)
            count += 1
        except DuplicateBookError:
            continue  # Skip duplicates
    
    return count


## 8) Duplicate book detection

Write `add_unique_book(catalog: Catalog, book: Book)` that raises `DuplicateBookError` if the ISBN is already present, else adds it.
Demonstrate `try/except` around this call to log a brief message and continue.

In [ ]:
def add_unique_book(catalog: Catalog, book: Book) -> None:
    """
    Raises DuplicateBookError if ISBN already present, else adds it.
    """
    catalog.add_book(book)  # Will raise DuplicateBookError if duplicate

    

In [ ]:
# Your code here
def add_unique_book(catalog: Catalog, book: Book) -> None:
    ...

# Demo with try/except here
def demo_add_unique():
    cat = Catalog()
    book1 = Book(isbn="1234567890", title="Test Book")
    
    try:
        add_unique_book(cat, book1)
        print("Book added successfully")
    except DuplicateBookError as e:
        print(f"Duplicate detected: {e}")
    
    # Try adding same book again
    try:
        add_unique_book(cat, book1)
        print("Book added successfully")
    except DuplicateBookError as e:
        print(f"Duplicate detected: {e}")

## 9) Timeout handling (simulated)

Create `fetch_cover_image(isbn: str, timeout_s: float=0.1) -> bytes` that **simulates** a timeout by raising `TimeoutError` when `timeout_s` < 0.05.
Write `get_cover_or_none(isbn)` that calls it in `try/except TimeoutError` and returns `None` on timeout.

In [ ]:
# Your code here
def fetch_cover_image(isbn: str, timeout_s: float = 0.1) -> bytes:
    """
    Simulates a timeout by raising TimeoutError when timeout_s < 0.05.
    """
    if timeout_s < 0.05:
        raise TimeoutError(f"Request timed out after {timeout_s}s")
    return b"fake_image_data"
def get_cover_or_none(isbn: str):
    ...
def get_cover_or_none(isbn: str):
    """
    Returns cover image bytes or None on timeout.
    """
    try:
        return fetch_cover_image(isbn, timeout_s=0.01)
    except TimeoutError:
        return None


## 10) Sanitizing member IDs

Implement `sanitize_member_id(value) -> str` that:
- Raises `TypeError` if not `str`
- Strips spaces, uppercases
- Raises `ValueError` if final form is empty or contains non-alphanumeric chars

In [ ]:
# Your code here
def validate_copies(copies: int) -> int:
    """
    Validates and sanitizes member ID.
    - Raises TypeError if not str
    - Strips spaces, uppercases
    - Raises ValueError if empty or contains non-alphanumeric chars
    """
    if not isinstance(value, str):
        raise TypeError("member_id must be a string")
    
    sanitized = value.strip().upper()
    
    if not sanitized:
        raise ValueError("member_id cannot be empty")
    
    if not sanitized.isalnum():
        raise ValueError("member_id must contain only alphanumeric characters")
    
    return sanitized
    ...

## 11) Convert asserts to exceptions

Given legacy code that uses `assert copies >= 0`, replace it with explicit `if` + `raise ValueError('copies must be non-negative')` in a function `validate_copies(copies: int) -> int` that returns the validated value.

In [ ]:
# Your code here
def validate_copies(copies: int) -> int:
    """
    Validates copies value and returns it.
    Raises ValueError instead of using assert.
    """
    if copies < 0:
        raise ValueError("copies must be non-negative")
    return copies
    ...

## 12) Basic unit tests for `validate_isbn`

Create a `tests/`-style cell using `unittest` that verifies:
- Valid 10- and 13-digit ISBNs pass
- Bad inputs raise `ValueError`
Do **not** import any third-party packages.

In [ ]:
# Your code here
import unittest

class TestValidateISBN(unittest.TestCase):
    def test_valid_10_and_13(self):
        """Test valid 10 and 13 digit ISBNs."""
        # Valid 10-digit
        result = validate_isbn("1234567890")
        self.assertEqual(result, "1234567890")
        
        # Valid 13-digit
        result = validate_isbn("9781402894626")
        self.assertEqual(result, "9781402894626")
        
        # With hyphens
        result = validate_isbn("978-1-4028-9462-6")
        self.assertEqual(result, "9781402894626")
    
    def test_invalid_values(self):
        """Test that invalid ISBNs raise ValueError."""
        with self.assertRaises(ValueError):
            validate_isbn("123")  # Too short
        
        with self.assertRaises(ValueError):
            validate_isbn("12345678901234")  # Wrong length
        
        with self.assertRaises(ValueError):
            validate_isbn("123-ABC-7890")  # Contains letters

# if __name__ == '__main__':
#     unittest.main(argv=['-v'], exit=False)  # Uncomment to run in notebook

## 13) Unit test with `assertRaises`

Write a `unittest.TestCase` verifying that adding a duplicate ISBN to a `Catalog` raises `DuplicateBookError`.
Use `setUp` to create a fresh `Catalog` and seed it with one book.

In [ ]:
# Your code here
import unittest
class TestCatalogDuplicates(unittest.TestCase):
    def setUp(self):
        """Create fresh catalog with one book."""
        self.catalog = Catalog()
        self.book = Book(isbn="1234567890", title="Test Book")
        self.catalog.add_book(self.book)
    
    def test_duplicate_raises(self):
        """Test that adding duplicate ISBN raises DuplicateBookError."""
        duplicate = Book(isbn="1234567890", title="Another Book")
        with self.assertRaises(DuplicateBookError):
            self.catalog.add_book(duplicate)
            
# if __name__ == '__main__':
#     unittest.main(argv=['-v'], exit=False)

## 14) Subtests for multiple bad IDs (optional pattern)

Using `unittest`, write a single test method that loops over a collection of **invalid** member IDs and, using `self.subTest`, asserts that `sanitize_member_id` raises `ValueError` for each.

In [ ]:
# Your code here
import unittest

class TestMemberIdSanitization(unittest.TestCase):
    def test_bad_ids(self):
        """Test multiple invalid member IDs using subtests."""
        bad_values = ["", "   ", "abc!", "id with space", "###"]
        for bad_id in bad_values:
            with self.subTest(bad_id=bad_id):
                with self.assertRaises(ValueError):
                    sanitize_member_id(bad_id)

# if __name__ == '__main__':
#     unittest.main(argv=['-v'], exit=False)

## 15) try/except/else logging

Write `try_index_book(catalog: Catalog, book: Book) -> bool` that
- Tries to `add_book`
- On exception, returns `False`
- Uses `else` to return `True` only when no exception occurred

In [ ]:
# Your code here
def try_index_book(catalog: Catalog, book: Book) -> bool:
    """
    Tries to add book. Returns True on success, False on exception.
    Uses else clause for success case.
    """
    try:
        catalog.add_book(book)
    except (DuplicateBookError, ValueError):
        return False
    else:
        return True
    ...

## 16) Graceful KeyboardInterrupt

Create `interactive_copies_prompt()` that repeatedly prompts the user for copies (use `input()`), converts using `parse_copies`, and prints the value.
If the user hits `Ctrl+C`, catch `KeyboardInterrupt` and print `Bye!` before returning.

In [ ]:
# Your code here
def interactive_copies_prompt():
    """
    Repeatedly prompts user for copies. Handles Ctrl+C gracefully.
    """
    try:
        while True:
            user_input = input("Enter number of copies (Ctrl+C to quit): ")
            try:
                copies = parse_copies(user_input)
                print(f"Valid copies: {copies}")
            except ValueError as e:
                print(f"Error: {e}")
    except KeyboardInterrupt:
        print("\nBye!")

# Note: Be mindful running this in notebooks; it's acceptable to provide the function without calling it.

## 17) Catalog validator that accumulates errors

Implement `validate_catalog_items(items: list[dict]) -> list[str]` that walks a list of book dicts and **collects** error messages rather than raising immediately.
Return a list of error strings (empty if valid).

In [ ]:
# Your code here
def validate_catalog_items(items: list[dict]) -> list[str]:
    """
    Validates list of book dicts. Returns list of error messages.
    """
    errors = []
    
    for i, item in enumerate(items):
        # Check required fields
        if "isbn" not in item:
            errors.append(f"Item {i}: missing 'isbn' field")
        elif not isinstance(item["isbn"], str):
            errors.append(f"Item {i}: 'isbn' must be a string")
        else:
            # Validate ISBN format
            try:
                validate_isbn(item["isbn"])
            except ValueError as e:
                errors.append(f"Item {i}: {e}")
        
        if "title" not in item:
            errors.append(f"Item {i}: missing 'title' field")
        elif not isinstance(item["title"], str):
            errors.append(f"Item {i}: 'title' must be a string")
        
        if "copies" in item:
            try:
                copies = int(item["copies"])
                if copies < 0:
                    errors.append(f"Item {i}: copies must be non-negative")
            except (ValueError, TypeError):
                errors.append(f"Item {i}: copies must be an integer")
    
    return errors
    ...

## 18) Replace broad except

Refactor a function that currently uses a bare `except:` to instead catch **only** `ValueError` and `TypeError` and re-raise unknown exceptions. Provide a before/after example and explain why broad except is risky.

In [ ]:
# Your code here
def before_style(x):
    """BAD: Swallows all exceptions including KeyboardInterrupt."""
    try:
        return int(x)
    except:
        return 0

def after_style(x):
    try:
        return int(x)
    except (ValueError, TypeError):
        return 0

# Add a short explanation in comments.

# GOOD: Only catches ValueError, allowing KeyboardInterrupt to propagate.
# This prevents the program from ignoring user interrupts.
# Hides programming errors.



## 19) Exit codes on failure

Write `main(argv)` that attempts to load a catalog JSON path from `argv[1]` using `safe_load_catalog`.
- Return `0` on success, `2` on `FileNotFoundError`, `3` on `json.JSONDecodeError`, otherwise `1`.
Use `sys.exit(main(sys.argv))` pattern in a protected `if __name__ == '__main__':` block (comment it out in notebook).

In [ ]:
def main(argv: list[str]) -> int:
    """
    Loads catalog from JSON path in argv[1].
    Returns: 0 on success, 2 on FileNotFoundError, 
             3 on JSONDecodeError, 1 on other errors.
    """
    if len(argv) < 2:
        print("Usage: python script.py <catalog.json>")
        return 1
    
    path = argv[1]
    catalog = Catalog()
    
    try:
        count = safe_load_catalog(path, catalog)
        print(f"Successfully loaded {count} books")
        return 0
    except FileNotFoundError:
        print(f"Error: File not found: {path}")
        return 2
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON in file: {path}")
        return 3
    except Exception as e:
        print(f"Unexpected error: {e}")
        return 1

# if __name__ == '__main__':
#     sys.exit(main(sys.argv))

## 20) End-to-end happy path test

Using `unittest`, write a test that:
- Builds a fresh `Catalog`
- Adds a valid `Book`
- Creates a `Loan` due tomorrow and asserts `assert_not_overdue` returns `True`
This is a **basic** end-to-end happy path—no Week 8+ features.

In [ ]:
# Your code here
import unittest
from datetime import datetime, timedelta

class TestHappyPath(unittest.TestCase):
    def test_happy_flow(self):
        """Test complete happy path: add book, create loan, check not overdue."""
        # Build fresh catalog
        catalog = Catalog()
        
        # Add valid book
        book = Book(isbn="9781234567890", title="Python Testing", copies=3)
        catalog.add_book(book)
        
        # Verify book was added
        retrieved = catalog.get_book("9781234567890")
        self.assertIsNotNone(retrieved)
        self.assertEqual(retrieved.title, "Python Testing")
        
        # Create loan due tomorrow
        tomorrow = datetime.now() + timedelta(days=1)
        loan = Loan(isbn="9781234567890", member_id="M001", due_date=tomorrow)
        
        # Assert loan is not overdue
        result = assert_not_overdue(loan)
        self.assertTrue(result)

        ...

if __name__ == '__main__':
     print("All solutions implemented!")
     print("\nTo run tests, uncomment the unittest.main() calls in each test class.")
#     unittest.main(argv=['-v'], exit=False)

## Python skills you'll need (Weeks 1–7)

- **Core syntax & data types:** variables, strings, ints/floats, booleans
- **Collections:** lists, dicts (basic use only)
- **Control flow:** `if/elif/else`, `for` loops, `while` loops
- **Functions & modules:** defining functions, parameters, returns, imports
- **File I/O:** open/read/write text and JSON (basic)
- **Classes & objects:** defining simple classes, `__init__`, instance methods, attributes
- **Encapsulation basics:** private attributes (naming convention), validation via methods/properties
- **Methods:** instance/class/static methods (as introduced in Week 6)
- **Error handling:** `try` / `except` / `else` / `finally`, `raise`, custom exceptions (Week 7)
- **Basic testing:** `unittest.TestCase`, `assertRaises`, `setUp`/`tearDown`, `subTest` (Week 7)
- **Standard library familiarity:** `datetime`, `json`, `sys`, built-in exceptions
